# Book recommendation System

- Web scraping: Popular lists from Goodreads, link: https://www.goodreads.com/list/popular_lists
- API retrieval: OpenLibrary, link: https://openlibrary.org/developers/api
* information on API usage find here: https://openlibrary.org/dev/docs/api/search

Information to extract from each source:

Book name, author, raiting in stars, language, published date, pages, ISBN 10, genres/subjects (goodreads/openlibrary)

## 1. Web scraping: Goodreads

In [1]:
# 📚 Basic libraries
import pandas as pd
import os
import random
import re
from bs4 import BeautifulSoup

#❗New Libraries !
import time
from getpass import getpass # to safely storage your password
# selenium libraries
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import pathlib

In [2]:
# ⚙️ Settings
pd.set_option('display.max_columns', None) # display all columns
import warnings
warnings.filterwarnings('ignore') # ignore warnings

In [4]:
driver= webdriver.Chrome(service=Service(ChromeDriverManager().install()))
url='https://www.goodreads.com/list/popular_lists'
# open the website
driver.get(url)

In [5]:
#get the names of the first available lists from the loaded webpage:

list_links = driver.find_elements(By.CLASS_NAME, 'listTitle')
lists_found=[]

for link in list_links:
    name=link.text
    href=link.get_attribute('href')
    lists_found.append({'name':name,'url':href})
for i, l in enumerate(lists_found):
    print(f"{i}: {l['name']}")


0: Best Books Ever
1: Books That Everyone Should Read At Least Once
2: Best Young Adult Books
3: Books That Should Be Made Into Movies
4: Best Books of the 20th Century
5: Best Book Boyfriends
6: Best Books of the Decade: 2000s
7: The Best Epic Fantasy (fiction)
8: Best Dystopian and Post-Apocalyptic Fiction
9: Best Historical Fiction


In [7]:
lists_found

[{'name': 'Best Books Ever',
  'url': 'https://www.goodreads.com/list/show/1.Best_Books_Ever'},
 {'name': 'Books That Everyone Should Read At Least Once',
  'url': 'https://www.goodreads.com/list/show/264.Books_That_Everyone_Should_Read_At_Least_Once'},
 {'name': 'Best Young Adult Books',
  'url': 'https://www.goodreads.com/list/show/43.Best_Young_Adult_Books'},
 {'name': 'Books That Should Be Made Into Movies',
  'url': 'https://www.goodreads.com/list/show/1043.Books_That_Should_Be_Made_Into_Movies'},
 {'name': 'Best Books of the 20th Century',
  'url': 'https://www.goodreads.com/list/show/6.Best_Books_of_the_20th_Century'},
 {'name': 'Best Book Boyfriends',
  'url': 'https://www.goodreads.com/list/show/10762.Best_Book_Boyfriends'},
 {'name': 'Best Books of the Decade: 2000s',
  'url': 'https://www.goodreads.com/list/show/5.Best_Books_of_the_Decade_2000s'},
 {'name': 'The Best Epic Fantasy (fiction)',
  'url': 'https://www.goodreads.com/list/show/50.The_Best_Epic_Fantasy_fiction_'},
 

In [9]:
#Ask the user which list wants to see:
choice=int(input('Pick a list index: '))
driver.get(lists_found[choice]['url'])

Pick a list index:  1


In [12]:
#as the page is HTML we will use Beautifulsoup to scrape:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/123.0 Safari/537.36"
}
list_url= lists_found[choice]['url']
r = requests.get(list_url, headers=headers)
print(r.status_code)

200


In [13]:
html=r.content
soup=BeautifulSoup(html,'html.parser')

In [15]:
#finding the links to each book:
soup.find_all